In [1]:
import json
import pandas as pd


In [2]:
def load_jsonl_sweep(path):
    """Load one row per hyperparameter config from the sweep script's jsonl.
    Each jsonl record has {'config': {...}, 'log10_mse': float, 'spearmanr': float},
    with metrics already averaged across K folds. We convert to the column layout
    that the plotting code expects ('Test Loss', 'Spearman - Test', etc.) and
    drop the sweep's built-in OLS baseline — the notebook appends its own
    RidgeCV 'additive' rows downstream."""
    rows = []
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            cfg = rec['config']
            if cfg.get('num_layers') == 'OLS':
                continue
            rows.append({
                'epochs':        cfg['epochs'],
                'optimizer':     cfg['optimizer'],
                'batch_size':    cfg['batch_size'],
                'layer_size':    cfg['layer_size'],
                'num_layers':    cfg['num_layers'],
                'learning_rate': cfg['learning_rate'],
                'Test Loss':       10.0 ** rec['log10_mse'],
                'Val Loss':        float('nan'),
                'Spearman - Val':  float('nan'),
                'Spearman - Test': rec['spearmanr'],
            })
    return pd.DataFrame(rows)


In [ ]:
# Load sweep results (log2 training target only) from the jsonl files written by
# analysis/multievolve_hyperparameter_sweep.py
df_apex_log2   = load_jsonl_sweep('../output/apex/apex_log2_hparams_results.jsonl')
df_dcasrx_log2 = load_jsonl_sweep('../output/dcasrx/dcasrx_log2_hparams_results.jsonl')
df_huabc2_expr_log2 = load_jsonl_sweep('../output/huabc2/huabc2_expr_log2_hparams_results.jsonl')
df_huabc2_bind_log2 = load_jsonl_sweep('../output/huabc2/huabc2_bind_log2_hparams_results.jsonl')
print(f'APEX:          {len(df_apex_log2)} configs')
print(f'dCasRx:        {len(df_dcasrx_log2)} configs')
print(f'HuABC2 (expr): {len(df_huabc2_expr_log2)} configs')
print(f'HuABC2 (bind): {len(df_huabc2_bind_log2)} configs')

## Add ridge (additive) baseline

Train RidgeCV (`sklearn`) on the same K-fold splits and one-hot features used
by the MULTI-evolve NN sweep, on log$_2$ fold change. Append results to each
DataFrame so they appear alongside the NN configurations in the boxplots.


In [ ]:
import os, sys, numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error

REPO = os.path.join(os.path.dirname(os.getcwd()), 'MULTI-evolve')
sys.path.insert(0, REPO)

from multievolve.splitters import KFoldProteinSplitter
from multievolve.featurizers import OneHotFeaturizer

ROOT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT_DIR, 'data')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')

# Fine-grained alpha grid: 50 points log-spaced over [10^-3, 10^3]
RIDGE_ALPHAS = np.logspace(-3, 3, 50)


def compute_ridge_results(protein, n_splits=5, measurement=None):
    """Train RidgeCV on the same K-fold splits and one-hot features used by the
    MULTI-evolve NN sweep (on log2-transformed fold change). Returns rows
    compatible with the sweep DataFrames.

    Parameters
    ----------
    protein : str
        Protein name ('apex', 'dcasrx', or 'huabc2').
    n_splits : int
        Number of cross-validation folds.
    measurement : str or None
        Required for huabc2: 'expr' or 'bind'.
    """
    proteins_dir = os.path.join(OUTPUT_DIR, protein)

    if protein == 'huabc2':
        assert measurement is not None, '--measurement required for huabc2'
        training_csv = os.path.join(
            proteins_dir, f'huabc2_training_{measurement}_log2.csv')
        # Multi-chain: list of FASTA files
        wt_fasta = [
            os.path.join(DATA_DIR, 'huabc2_vh.fasta'),
            os.path.join(DATA_DIR, 'huabc2_vl.fasta'),
        ]
    else:
        training_csv = os.path.join(
            proteins_dir, f'{protein}_training_log2.csv')
        wt_fasta = os.path.join(DATA_DIR, f'{protein}.fasta')

    splitter = KFoldProteinSplitter(
        protein, training_csv, wt_fasta,
        csv_has_header=True, use_cache=True, y_scaling=True, val_split=0.15,
    )
    splits = splitter.generate_splits(n_splits=n_splits)
    featurizer = OneHotFeaturizer(protein=protein, use_cache=True)

    rows = []
    for i, split in enumerate(splits):
        X_train = featurizer.featurize(split.splits['X_train']).reshape(len(split.splits['X_train']), -1)
        X_test  = featurizer.featurize(split.splits['X_test']).reshape(len(split.splits['X_test']), -1)
        y_train = np.array(split.splits['y_train'])
        y_test  = np.array(split.splits['y_test'])

        model = RidgeCV(alphas=RIDGE_ALPHAS, fit_intercept=False)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        sr, _ = spearmanr(y_test, y_pred)

        rows.append({
            'name': f'additive-fold-{i}',
            'epochs': 0, 'optimizer': 'ridge',
            'batch_size': 0, 'layer_size': 0,
            'num_layers': 'additive', 'learning_rate': 0,
            'Test Loss': mse,
            'Val Loss': np.nan,
            'Spearman - Val': np.nan,
            'Spearman - Test': sr,
            'ridge_alpha': model.alpha_,
        })

    return pd.DataFrame(rows)


# Compute and append Ridge rows (labeled "additive") to each log2 DataFrame.
print(f"Ridge alpha grid: {len(RIDGE_ALPHAS)} values in [{RIDGE_ALPHAS[0]:.4g}, {RIDGE_ALPHAS[-1]:.4g}]")
for protein, df_name, meas in [
    ('apex',   'df_apex_log2',          None),
    ('dcasrx', 'df_dcasrx_log2',        None),
    ('huabc2', 'df_huabc2_expr_log2',   'expr'),
    ('huabc2', 'df_huabc2_bind_log2',   'bind'),
]:
    ridge_df = compute_ridge_results(protein, measurement=meas)
    target = globals()[df_name]
    globals()[df_name] = pd.concat([target, ridge_df], ignore_index=True)
    per_fold = ', '.join([f'{a:.4g}' for a in ridge_df['ridge_alpha']])
    print(f"{df_name}: MSE={ridge_df['Test Loss'].mean():.4f}, "
          f"Spearman={ridge_df['Spearman - Test'].mean():.3f}, "
          f"alpha per fold=[{per_fold}]")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import os

os.makedirs('../figures', exist_ok=True)

ADDITIVE_LABEL = 'additive'

def plot_hparams_results(raw_df, experiment, split="Test", display_name=None):

    raw_df[f"log10 {split} Loss"] = np.log10(raw_df[f"{split} Loss"])

    # Separate additive model from NN results
    is_additive = raw_df["num_layers"] == ADDITIVE_LABEL
    additive_raw = raw_df[is_additive].copy()
    nn_raw = raw_df[~is_additive]

    # compute average across runs with same hparams (NN only)
    metrics_cols = ['Test Loss', 'Val Loss', 'Spearman - Val', 'Spearman - Test', f"log10 {split} Loss"]
    hparams_cols = ['epochs', 'layer_size', 'batch_size', 'num_layers', 'learning_rate']
    data = []
    for name, group_df in nn_raw.groupby(hparams_cols):
        row_hparams = {hparam: group_df[hparam].values[0] for hparam in hparams_cols}
        row_metrics = {metric: np.mean(group_df[metric].values) for metric in metrics_cols}
        data.append({**row_hparams, **row_metrics})
    df = pd.DataFrame(data)

    # Additive model values (averaged across folds)
    additive_vals = {}
    if len(additive_raw) > 0:
        for metric in metrics_cols:
            additive_vals[metric] = np.mean(additive_raw[metric].dropna().values) if metric in additive_raw.columns else np.nan

    # Varying hparams
    hparams = [
        ("num_layers", "Layers"),
        ("batch_size", "Batch size"),
        ("learning_rate", "Learning rate"),
    ]
    metrics = [
        (f"log10 {split} Loss", r"$\log_{10}(\mathrm{MSE})$"),
        (f"Spearman - {split}", "Spearman correlation"),
    ]
    
    # Best NN models — split by 0 hidden layers vs >=1 hidden layers
    df["num_layers_int"] = df["num_layers"].astype(int)
    zero_df  = df[df["num_layers_int"] == 0]
    multi_df = df[df["num_layers_int"] >= 1]
    best_mse_multi = multi_df.loc[multi_df[f"log10 {split} Loss"].idxmin()] if len(multi_df) else None
    best_sr_multi  = multi_df.loc[multi_df[f"Spearman - {split}"].idxmax()] if len(multi_df) else None
    best_mse_zero  = zero_df.loc[zero_df[f"log10 {split} Loss"].idxmin()]   if len(zero_df)  else None
    best_sr_zero   = zero_df.loc[zero_df[f"Spearman - {split}"].idxmax()]   if len(zero_df)  else None
    
    # Colors
    box_color = "#9b7abf"
    additive_color = "#e07b54"
    
    fig, axes = plt.subplots(2, 3, figsize=(11, 8.5))
    
    for row, (metric_col, metric_label) in enumerate(metrics):
        for col, (hparam_col, hparam_label) in enumerate(hparams):
            ax = axes[row, col]

            groups = sorted(df[hparam_col].unique())
            data_groups = [df.loc[df[hparam_col] == g, metric_col].dropna().values for g in groups]
    
            bp = ax.boxplot(
                data_groups,
                tick_labels=[str(g) for g in groups],
                patch_artist=True,
                widths=0.55,
                showfliers=True,
                whis=[15, 85],
                flierprops=dict(marker="o", markersize=4, markerfacecolor="gray",
                                markeredgecolor="gray", alpha=0.6),
                medianprops=dict(color="black", linewidth=1.5),
                whiskerprops=dict(color="black"),
                capprops=dict(color="black"),
            )
            for patch in bp["boxes"]:
                patch.set_facecolor(box_color)
                patch.set_edgecolor("black")

            # Additive horizontal line
            if metric_col in additive_vals and not np.isnan(additive_vals[metric_col]):
                ax.axhline(additive_vals[metric_col], color=additive_color, linewidth=2,
                           linestyle='--', zorder=5)

            ax.set_xlabel(hparam_label, fontsize=11)
            if col == 0:
                ax.set_ylabel(metric_label, fontsize=11)
    
            # y-axis limits
            if metric_col == f"Spearman - {split}":
                ax.set_ylim(-0.05, 1.05)

    # Figure-level legend
    legend_elements = [
        Line2D([0], [0], color=additive_color, linewidth=2, linestyle='--', label='additive'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=1, fontsize=10,
               frameon=True, fancybox=True, edgecolor='gray')
    
    # Annotation text
    def _model_desc(row):
        return (f"Layers: {int(row['num_layers'])}, "
                f"Batch size: {int(row['batch_size'])}, "
                f"LR: {row['learning_rate']}")

    def _block(title, row):
        if row is None:
            return f"{title}: n/a"
        return (f"{title}:\n"
                f"  {_model_desc(row)}\n"
                f"  $\\log_{{10}}$MSE = {row[f'log10 {split} Loss']:.2f}, "
                f"Spearman = {row[f'Spearman - {split}']:.3f}")

    anno_add = (
        f"Additive (Ridge):\n"
        f"  $\\log_{{10}}$MSE = {additive_vals.get(f'log10 {split} Loss', float('nan')):.2f}, "
        f"Spearman = {additive_vals.get(f'Spearman - {split}', float('nan')):.3f}"
    )
    anno = "\n\n".join([
        anno_add,
        _block(r"Best NN, $\geq$1 layers (lowest MSE)", best_mse_multi),
        _block(r"Best NN, 0 layers (lowest MSE)",       best_mse_zero),
        _block(r"Best NN, $\geq$1 layers (highest Spearman)", best_sr_multi),
        _block(r"Best NN, 0 layers (highest Spearman)",       best_sr_zero),
    ])
    
    fig.text(0.02, 0.99, f"Dataset: {display_name or experiment} - {split}", fontsize=13, fontweight="bold",
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#c8d4e8", edgecolor="gray"))
    
    fig.text(0.98, 0.99, anno, fontsize=8, fontfamily="monospace",
            ha="right", va="top")
    
    plt.tight_layout(rect=[0, 0.04, 1, 0.78])

    # Save to PDF
    safe_name = experiment.lower().replace(' ', '_').replace('(', '').replace(')', '')
    fig.savefig(f'../figures/hparams_{safe_name}.pdf', bbox_inches='tight')
    plt.show()


plot_hparams_results(df_apex_log2,          'APEX',             split="Test", display_name='APEX ($-$)A134P')
plot_hparams_results(df_dcasrx_log2,        'dCasRx',           split="Test")
plot_hparams_results(df_huabc2_expr_log2,   'HuABC2 (expr)',    split="Test")
plot_hparams_results(df_huabc2_bind_log2,   'HuABC2 (bind)',    split="Test")

In [ ]:
# Simplified figure for main.tex:
# Only the "Layers" sweep, only log2 datasets.
# Rows: log10 Test Loss (top), Spearman - Test (bottom).
# Cols: APEX, dCasRx, HuABC2 (expr), HuABC2 (bind).

def plot_layers_log2(dataset_dfs, split="Test"):
    """
    Parameters
    ----------
    dataset_dfs : list of (label, DataFrame) tuples
    """
    box_color = "#9b7abf"
    additive_color = "#e07b54"

    metrics = [
        (f"log10 {split} Loss", r"$\log_{10}(\mathrm{MSE})$"),
        (f"Spearman - {split}", "Spearman correlation"),
    ]

    n_datasets = len(dataset_dfs)
    fig, axes = plt.subplots(2, n_datasets,
                             figsize=(2 * n_datasets, 5),
                             sharex=True, sharey="row")
    if n_datasets == 1:
        axes = axes.reshape(2, 1)

    for col, (dataset_label, raw_df) in enumerate(dataset_dfs):
        raw_df = raw_df.copy()
        raw_df[f"log10 {split} Loss"] = np.log10(raw_df[f"{split} Loss"])

        is_additive = raw_df["num_layers"] == ADDITIVE_LABEL
        additive_raw = raw_df[is_additive].copy()
        nn_raw = raw_df[~is_additive].copy()

        metrics_cols = ["Test Loss", "Val Loss", "Spearman - Val", "Spearman - Test",
                        f"log10 {split} Loss"]
        hparams_cols = ["epochs", "layer_size", "batch_size", "num_layers", "learning_rate"]
        data = []
        for _, group_df in nn_raw.groupby(hparams_cols):
            row_h = {h: group_df[h].values[0] for h in hparams_cols}
            row_m = {m: np.mean(group_df[m].values) for m in metrics_cols}
            data.append({**row_h, **row_m})
        df = pd.DataFrame(data)

        additive_vals = {}
        for metric in metrics_cols:
            if metric in additive_raw.columns:
                vals = additive_raw[metric].dropna().values
                additive_vals[metric] = float(np.mean(vals)) if len(vals) else float("nan")
            else:
                additive_vals[metric] = float("nan")

        df["num_layers"] = df["num_layers"].astype(int)
        groups = sorted(df["num_layers"].unique())

        for row, (metric_col, metric_label) in enumerate(metrics):
            ax = axes[row, col]
            data_groups = [df.loc[df["num_layers"] == g, metric_col].dropna().values
                           for g in groups]

            bp = ax.boxplot(
                data_groups,
                tick_labels=[str(g) for g in groups],
                patch_artist=True,
                widths=0.55,
                showfliers=True,
                whis=[15, 85],
                flierprops=dict(marker="o", markersize=4, markerfacecolor="gray",
                                markeredgecolor="gray", alpha=0.6),
                medianprops=dict(color="black", linewidth=1.5),
                whiskerprops=dict(color="black"),
                capprops=dict(color="black"),
            )
            for patch in bp["boxes"]:
                patch.set_facecolor(box_color)
                patch.set_edgecolor("black")

            if not np.isnan(additive_vals.get(metric_col, float("nan"))):
                ax.axhline(additive_vals[metric_col], color=additive_color,
                           linewidth=2, linestyle="--", zorder=5)

            if row == 0:
                ax.set_title(dataset_label, fontsize=12)
            if row == 1:
                ax.set_xlabel("Layers", fontsize=11)
            if col == 0:
                ax.set_ylabel(metric_label, fontsize=11)

            if metric_col == f"Spearman - {split}":
                ax.set_ylim(-0.05, 1.05)

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    fig.savefig("../figures/layers_log2.pdf", bbox_inches="tight")
    plt.show()


plot_layers_log2([
    ("APEX ($-$)A134P", df_apex_log2),
    ("dCasRx",         df_dcasrx_log2),
    ("HuABC2 (expr)",  df_huabc2_expr_log2),
    ("HuABC2 (bind)",  df_huabc2_bind_log2),
], split="Test")